In [2]:
import pandas as pd
from datasets import load_dataset
from collections import defaultdict
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

# Quran + OT/NT adjustments

In [330]:
# dataset = load_dataset("CAMeL-Lab/BAREC-Corpus-v1.0")
# print(os.listdir("../../../Seg/ReconstructingDocs/processed_data/Final/all_data.tsv"))
dataset = pd.read_csv("../../../Seg/ReconstructingDocs/processed_data/Final/all_data_2.tsv", sep="\t")

In [331]:
split_mapping = {
    "Train": "train",
    "Dev": "dev",
    "Test": "test",
    "train": "train",
    "dev": "dev",
    "test": "test",
    "No-Split": "No-Split",
    "No Split": "No-Split",
    "STask": "STask"
}
dataset["new_split"] = dataset["split"].map(split_mapping)

In [332]:
NT_BOOKS = ["Matthew", "Mark", "Luke"]
OT_BOOKS = ["Exodus", "Genesis"]
def correct_nt_ot_names(doc_name):
    # remove prefix
    doc_name = doc_name[3:]
    if "gen" in doc_name:
        doc_name = doc_name.replace("gen", "Genesis")
    return f"{doc_name}_001"
def map_split(doc_name):
    for book in NT_BOOKS:
        if book in doc_name:
            if f"NT_{doc_name}" in nt_ot_split_mapping:
                return nt_ot_split_mapping[f"NT_{doc_name}"]
    for book in OT_BOOKS:
        if book in doc_name:
            if f"OT_{doc_name}" in nt_ot_split_mapping:
                return nt_ot_split_mapping[f"OT_{doc_name}"]
    return "No-Split"
dataset.loc[(dataset["source"] == "New Testament") | (dataset["source"] == "Old Testament"), "doc_name"] = dataset.loc[(dataset["source"] == "New Testament") | (dataset["source"] == "Old Testament"), "doc_name"].apply(correct_nt_ot_names)
dataset.loc[(dataset["source"] == "New Testament") | (dataset["source"] == "Old Testament"), "new_split"] = dataset.loc[(dataset["source"] == "New Testament") | (dataset["source"] == "Old Testament"), "doc_name"].apply(map_split)
# dataset.loc[(dataset["source"] == "Quran")]

In [333]:
dataset[dataset["source"] == "New Testament"]["new_split"].value_counts()

new_split
No-Split    2587
train        457
dev          104
test          78
Name: count, dtype: int64

In [334]:
dataset.loc[((dataset["source"] == "New Testament"))].groupby(["new_split"]).agg({"word_count": "sum"}).reset_index()

,new_split,word_count
0,No-Split,46694
1,dev,1900
2,test,1423
3,train,8164


In [335]:
from random import random, seed
target_ratio = {
    "train": 0.7,
    "dev": 0.15,
    "test": 0.15
}
seed(102372831)
def split_by_wc(df):
    total_words = df["word_count"].sum()
    df = df.sample(frac=1.0, random_state=102372831).reset_index(drop=True)
    target_words = {k: v * total_words for k, v in target_ratio.items()}
    print(target_words)
    splits = {k: [] for k in target_ratio.keys()}
    curr_words = {k: 0 for k in target_ratio.keys()}
    for idx, row in df.iterrows():
        wc = row["word_count"]
        best_split = None
        best_score = None
        for split in target_words:
            remaining = target_words[split] - curr_words[split]
            score = remaining
            if best_score is None or score > best_score:
                best_score = score
                best_split = split
        splits[best_split].append(row["doc_name"])
        curr_words[best_split] += wc
    # add remaining docs to train
    return splits, curr_words, target_words

nt_no_split_assignments, _, _ = split_by_wc(dataset.loc[((dataset["source"] == "New Testament")) & (dataset["new_split"] == "No-Split")].groupby(["doc_name"]).agg({"word_count": "sum"}).reset_index())
ot_no_split_assignments, _, _ = split_by_wc(dataset.loc[((dataset["source"] == "Old Testament")) & (dataset["new_split"] == "No-Split")].groupby(["doc_name"]).agg({"word_count": "sum"}).reset_index())
quran_no_split_assignments, _, _ = split_by_wc(dataset.loc[((dataset["source"] == "Quran")) & (dataset["new_split"] == "No-Split")].groupby(["doc_name"]).agg({"word_count": "sum"}).reset_index())

{'train': 32685.8, 'dev': 7004.099999999999, 'test': 7004.099999999999}
{'train': 33098.799999999996, 'dev': 7092.599999999999, 'test': 7092.599999999999}
{'train': 67302.2, 'dev': 14421.9, 'test': 14421.9}


In [336]:
assignment_nt_no_split = {}
for k, v in nt_no_split_assignments.items():
    for doc in v:
        assignment_nt_no_split[doc] = k
assignment_ot_no_split = {}
for k, v in ot_no_split_assignments.items():
    for doc in v:
        assignment_ot_no_split[doc] = k

In [337]:
dataset.loc[((dataset["source"] == "New Testament")) & (dataset["new_split"] == "No-Split"), "doc_name"].apply(lambda x: assignment_nt_no_split[x])

941966    train
941967    train
941968    train
941969    train
941970    train
          ...  
945148    train
945149    train
945150    train
945151    train
945152    train
Name: doc_name, Length: 2587, dtype: object

In [338]:
dataset.loc[((dataset["source"] == "New Testament")) & (dataset["new_split"] == "No-Split"),"new_split"] = dataset.loc[((dataset["source"] == "New Testament")) & (dataset["new_split"] == "No-Split"), "doc_name"].apply(lambda x: assignment_nt_no_split[x])
dataset.loc[((dataset["source"] == "Old Testament")) & (dataset["new_split"] == "No-Split"),"new_split"] = dataset.loc[((dataset["source"] == "Old Testament")) & (dataset["new_split"] == "No-Split"), "doc_name"].apply(lambda x: assignment_ot_no_split[x])

In [339]:
# group by doc and sum word count
nt_ot_dataset = dataset.loc[((dataset["source"] == "New Testament")) & (dataset["new_split"] == "No-Split")]
original_nt_dataset = dataset.loc[((dataset["source"] == "New Testament")) & (dataset["new_split"] != "No-Split")]
original_nt_dataset = original_nt_dataset.groupby("doc_name")["word_count"].sum().reset_index()
nt_ot_dataset = nt_ot_dataset.groupby("doc_name")["word_count"].sum().reset_index()
nt_ot_dataset

,doc_name,word_count


In [340]:
dataset[dataset["source"] == "New Testament"]["new_split"].value_counts()

new_split
train    2290
test      470
dev       466
Name: count, dtype: int64

In [341]:
list(dataset[dataset["source"] == "Quran"].groupby("doc_name")["word_count"].sum().reset_index()["doc_name"].unique())

['Quran_Abasa_80',
 'Quran_Ad-Duhaa_93',
 'Quran_Ad-Dukhan_44',
 'Quran_Adh-Dhariyat_51',
 "Quran_Al-'Adiyat_100",
 "Quran_Al-'Alaq_96",
 "Quran_Al-'Ankabut_29",
 "Quran_Al-'Asr_103",
 "Quran_Al-A'la_87",
 "Quran_Al-A'raf_7",
 'Quran_Al-Ahqaf_46',
 'Quran_Al-Ahzab_33',
 "Quran_Al-An'am_6",
 'Quran_Al-Anbya_21',
 'Quran_Al-Anfal_8',
 'Quran_Al-Balad_90',
 'Quran_Al-Baqarah_2',
 'Quran_Al-Bayyinah_98',
 'Quran_Al-Buruj_85',
 'Quran_Al-Fajr_89',
 'Quran_Al-Falaq_113',
 'Quran_Al-Fath_48',
 'Quran_Al-Fatihah_1',
 'Quran_Al-Fil_105',
 'Quran_Al-Furqan_25',
 'Quran_Al-Ghashiyah_88',
 'Quran_Al-Hadid_57',
 'Quran_Al-Hajj_22',
 'Quran_Al-Haqqah_69',
 'Quran_Al-Hashr_59',
 'Quran_Al-Hijr_15',
 'Quran_Al-Hujurat_49',
 'Quran_Al-Humazah_104',
 'Quran_Al-Ikhlas_112',
 'Quran_Al-Infitar_82',
 'Quran_Al-Insan_76',
 'Quran_Al-Inshiqaq_84',
 'Quran_Al-Isra_17',
 'Quran_Al-Jathiyah_45',
 'Quran_Al-Jinn_72',
 "Quran_Al-Jumu'ah_62",
 'Quran_Al-Kafirun_109',
 'Quran_Al-Kahf_18',
 'Quran_Al-Kawthar_108',
 

In [342]:
dataset.loc[(dataset["source"] == "Quran")]["new_split"].value_counts()

new_split
No-Split    6236
Name: count, dtype: int64

In [343]:
quran_mappings = {
    "Ali 'Imran_3": "dev",
    "Al-Baqarah": "train",
    "Al-Fatihah": "dev",
    "Al-Qari'ah": "test",
    "At-Takathur": "test",
    "Al-'Asr": "test",
    "Al-Humazah": "test",
    "Al-Fil": "test",
    "Quraysh": "test",
    "Al-Ma'un": "test",
    "Al-Kawthar": "test",
    "An-Nasr": "test",
    "Al-Masad": "test",
    "Al-Ikhlas": "test",
    "Al-Falaq": "test",
    "An-Nas": "test",
}
def apply_quran_mappings(doc_name):
    needed_name = doc_name.split("_")[1]
    if needed_name in quran_mappings:
        return quran_mappings[needed_name]
    return "No-Split"
dataset.loc[(dataset["source"] == "Quran") & (dataset["new_split"] == "No-Split"), "new_split"] = dataset.loc[(dataset["source"] == "Quran") & (dataset["new_split"] == "No-Split"), "doc_name"].apply(apply_quran_mappings)


In [344]:
target_ratio = {
    "train": 0.7,
    "dev": 0.15,
    "test": 0.15
}
seed(102372831)
def split_by_wc(df):
    total_words = df["word_count"].sum()
    df = df.sample(frac=1.0, random_state=102372831).reset_index(drop=True)
    target_words = {k: v * total_words for k, v in target_ratio.items()}
    print(target_words)
    splits = {k: [] for k in target_ratio.keys()}
    curr_words = {k: 0 for k in target_ratio.keys()}
    for idx, row in df.iterrows():
        wc = row["word_count"]
        best_split = None
        best_score = None
        for split in target_words:
            remaining = target_words[split] - curr_words[split]
            score = remaining
            if best_score is None or score > best_score:
                best_score = score
                best_split = split
        splits[best_split].append(row["doc_name"])
        curr_words[best_split] += wc
    # add remaining docs to train
    return splits, curr_words, target_words

quran_no_split_assignments, _, _ = split_by_wc(dataset.loc[((dataset["source"] == "Quran")) & (dataset["new_split"] == "No-Split")].groupby(["doc_name"]).agg({"word_count": "sum"}).reset_index())

assignment_quran_no_split = {}
for k, v in quran_no_split_assignments.items():
    for doc in v:
        assignment_quran_no_split[doc] = k
dataset.loc[((dataset["source"] == "Quran") & (dataset["new_split"] == "No-Split")), "new_split"] = dataset.loc[((dataset["source"] == "Quran") & (dataset["new_split"] == "No-Split")), "doc_name"].apply(lambda x: assignment_quran_no_split[x])



{'train': 62031.899999999994, 'dev': 13292.55, 'test': 13292.55}


In [51]:
import json
with open("../../../Seg/ReconstructingDocs/processed_data/Final/doc_to_split_mapping.json", "r") as f:
    doc_to_split_mapping = json.load(f)
nt_ot_split_mapping = {k[6:-4]: v for k, v in doc_to_split_mapping.items() if (k.startswith("BAREC_NT_") or k.startswith("BAREC_OT_"))}
nt_ot_split_mapping

{'NT_Matthew_10_001': 'train',
 'NT_Matthew_11_001': 'train',
 'NT_Matthew_12_001': 'train',
 'NT_Matthew_13_001': 'train',
 'NT_Matthew_14_001': 'train',
 'NT_Matthew_5_001': 'train',
 'NT_Matthew_6_001': 'train',
 'NT_Matthew_7_001': 'train',
 'NT_Matthew_8_001': 'train',
 'NT_Matthew_9_001': 'train',
 'OT_Genesis_10_001': 'train',
 'OT_Genesis_11_001': 'train',
 'OT_Genesis_12_001': 'train',
 'OT_Genesis_13_001': 'train',
 'OT_Genesis_14_001': 'train',
 'OT_Genesis_15_001': 'train',
 'OT_Genesis_16_001': 'train',
 'OT_Genesis_17_001': 'train',
 'OT_Genesis_18_001': 'train',
 'OT_Genesis_4_001': 'train',
 'OT_Genesis_5_001': 'train',
 'OT_Genesis_6_001': 'train',
 'OT_Genesis_7_001': 'train',
 'OT_Genesis_8_001': 'train',
 'OT_Genesis_9_001': 'train',
 'NT_Matthew_1_001': 'dev',
 'NT_Matthew_2_001': 'dev',
 'NT_Matthew_3_001': 'dev',
 'NT_Matthew_4_001': 'dev',
 'OT_Genesis_1_001': 'dev',
 'OT_Genesis_2_001': 'dev',
 'OT_Genesis_3_001': 'dev',
 'NT_Matthew_15_001': 'test',
 'NT_Matth

In [314]:
dataset["new_split"].value_counts()

new_split
No-Split    816342
train        99409
test         14879
dev          14016
STask         5663
Name: count, dtype: int64

In [284]:
dataset.to_csv("adjusted_splits.tsv", sep="\t")

In [345]:
diwan_df = dataset[dataset["source"] == "Diwan"].copy()

docs_by_word_count = (
    diwan_df
    .groupby(["doc_name"])
    .agg({"word_count": "sum"})
    .reset_index()
    .sort_values(by="word_count", ascending=False)
)
# split by word count
diwan_mapping, _, _ = split_by_wc(docs_by_word_count)

diwan_mapping_docs = {}
for k, v in diwan_mapping.items():
    for doc in v:
        diwan_mapping_docs[doc] = k

diwan_mapping_docs


{'train': 5411652.399999999, 'dev': 1159639.8, 'test': 1159639.8}


{'العصر-الايوبي_اللواح_من-لائمي-إن-سفحت-الدمع-من-مقلي_001': 'train',
 'العصر-الاموي_الفرزدق_لنا-عدد-يربي-على-عدد-الحصى_001': 'train',
 'العصر-الأندلسي_أبو-الحسن-الششتري_لقد-أنا-شيء-عجيب_001': 'train',
 'العصر-الايوبي_ابن-الساعاتي_ولقد-نزلت-من-الجزيرة-منزلاً_001': 'train',
 'العصر-العباسي_البحتري_خلته-لما-تبدى_001': 'train',
 'العصر-العباسي_المأمون_إني-وأنت-رضيعا-قهوة-لطفت_001': 'train',
 'العصر-الايوبي_المكزون-السنجاري_لوجودك-الجزئي-كليات-أجنا_001': 'train',
 'العصر-العباسي_السري-الرفاء_لنا-من-الدهر-خصم-لا-نغالبه_001': 'train',
 'العصر-العباسي_ابن-الرومي_يا-أيها-الهارب-من-دهره_001': 'train',
 'العصر-الاموي_يزيد-بن-الطثرية_قل-للبوادر-والأحلاف-ما-لكم_001': 'train',
 'العصر-العباسي_الوأواء-الدمشقي_رعى-الله-ليلا-ضل-عنه-صباحه_001': 'train',
 'العصر-المملوكي_لسان-الدين-بن-الخطيب_أنا-إلى-الله-وإنا-له_001': 'train',
 'العصر-الايوبي_ابن-المقرب-العيوني_بع-واسطاً-بالنأي-والهجر_001': 'train',
 'العصر-العباسي_أبو-الحسن-الجرجاني_ولو-تراني-وقد-ظفرت-به_001': 'train',
 'العصر-العباسي_ابن-نباتة-السعدي_ف

In [346]:
# map the splits
dataset.loc[dataset["source"] == "Diwan", "new_split"] = dataset.loc[dataset["source"] == "Diwan", "doc_name"].apply(lambda x: diwan_mapping_docs[x])

In [347]:
dataset[dataset["source"] == "Diwan"]["new_split"].value_counts()

new_split
train    571207
dev      122661
test     122474
Name: count, dtype: int64

In [348]:
dataset.to_csv("adjusted_splits.tsv", sep="\t")

In [362]:
dataset[(dataset["source"] == "Quran")]["doc_name"].unique()

array(['Quran_Ar-Rahman_55', 'Quran_Al-Fath_48', 'Quran_An-Nahl_16',
       'Quran_Al-Hadid_57', 'Quran_An-Naml_27', 'Quran_Al-Fil_105',
       "Quran_An-Nazi'at_79", 'Quran_Abasa_80', 'Quran_Al-Qalam_68',
       "Quran_Al-Ma'arij_70", 'Quran_Al-Insan_76', 'Quran_Al-Ahzab_33',
       'Quran_Al-Inshiqaq_84', 'Quran_Al-Hajj_22', 'Quran_Al-Fatihah_1',
       'Quran_Ghafir_40', 'Quran_Ash-Sharh_94', "Quran_Al-An'am_6",
       'Quran_Al-Mujadila_58', 'Quran_Adh-Dhariyat_51',
       'Quran_Al-Layl_92', 'Quran_Al-Hujurat_49', 'Quran_Az-Zukhruf_43',
       'Quran_Al-Jinn_72', 'Quran_Al-Munafiqun_63',
       'Quran_Al-Mursalat_77', 'Quran_Al-Ahqaf_46', 'Quran_Muhammad_47',
       'Quran_Al-Anbya_21', 'Quran_Al-Haqqah_69', 'Quran_Al-Masad_111',
       'Quran_Al-Furqan_25', 'Quran_Al-Mutaffifin_83',
       'Quran_Al-Baqarah_2', 'Quran_Al-Muddaththir_74',
       'Quran_Al-Bayyinah_98', "Quran_Ash-Shu'ara_26", 'Quran_Al-Mulk_67',
       'Quran_Az-Zumar_39', 'Quran_Al-Fajr_89', 'Quran_An-Najm_53',
 

In [ ]:
# adjust quran, edit splits and put them into one paragraph
# add subtitles and gpt


In [368]:
quran_baqarah_ayah_bounds = {
    "1": (1, 23),
    "2": (24, 39),
    "3": (40, 60),
    "4": (61, 75),
    "5": (76, 90),
    "6": (91, 105),
    "7": (106, 121),
    "20": (285, 286),
    "19": (274, 284),
    "18": (261, 273),
    "17": (251, 260),
    "16": (238, 250),
}

quran_baqarah_subset_splits = {
    "1": "dev",
    "2": "dev",
    "3": "dev",
    "4": "dev",
    "5": "dev",
    "6": "train",
    "7": "train",
    "20": "train",
    "19": "train",
    "18": "train",
    "17": "train",
    "16": "train",
}

for subset, (start, end) in quran_baqarah_ayah_bounds.items():
    new_doc_name = f"Quran_Al-Baqarah_2_{int(subset):03d}"
    dataset.loc[(dataset["source"] == "Quran") & (dataset["doc_name"] == f"Quran_Al-Baqarah_2") & (dataset["paragraph_id"] >= start) & (dataset["paragraph_id"] <= end), "doc_name"] = new_doc_name
    # adjust split
    dataset.loc[(dataset["source"] == "Quran") & (dataset["doc_name"] == new_doc_name), "new_split"] = quran_baqarah_subset_splits[subset]
    
    # dataset.loc[(dataset["source"] == "Quran") & (dataset["doc_name"] == f"Quran_Al-Baqarah_{subset}") & (dataset["paragraph_id"] >= start) & (dataset["paragraph_id"] <= end), "new_split"] = quran_baqarah_subset_splits[subset]

# dataset[dataset["source"] == "Quran"]["new_split"].value_counts()

# dataset[(dataset["source"] == "Quran") & (dataset["doc_name"] == "Quran_Al-Baqarah_2") & (dataset["paragraph_id"] >=76) & (dataset["paragraph_id"] <= 91)]

In [391]:
quran_imran_ayah_bounds = {
    "1": (1, 12),
    "2": (13, 21),
    "3": (22, 30),
    "4": (31, 41),
    "5": (42, 52),
    "6": (53, 64),
    "7": (65, 75),
    "8": (76, 84),
    "9": (85, 97),
    "10": (98, 108),
    "15": (155, 164),
    "16": (165, 175),
    "17": (176, 184),
    "18": (185, 195),
    "19": (196, 200)
}

quran_imran_subset_splits = {
    "1": "train",
    "2": "train",
    "3": "train",
    "4": "train",
    "5": "train",
    "6": "train",
    "7": "train",
    "8": "train",
    "9": "train",
    "10": "train",
    "15": "test",
    "16": "test",
    "17": "test",
    "18": "test",
    "19": "test"
}
for subset, (start, end) in quran_imran_ayah_bounds.items():
    new_doc_name = f"Quran_Ali 'Imran_3_{int(subset):03d}"
    dataset.loc[(dataset["source"] == "Quran") & (dataset["doc_name"] == f"Quran_Ali 'Imran_3") & (dataset["paragraph_id"] >= start) & (dataset["paragraph_id"] <= end), "doc_name"] = new_doc_name
    # adjust split
    dataset.loc[(dataset["source"] == "Quran") & (dataset["doc_name"] == new_doc_name), "new_split"] = quran_imran_subset_splits[subset]

In [392]:
dataset[(dataset["source"] == "Quran") & (dataset["new_split"] == "train")]["doc_name"].unique()

array(['Quran_Ar-Rahman_55', 'Quran_Al-Fath_48', 'Quran_An-Nahl_16',
       'Quran_Al-Hadid_57', "Quran_An-Nazi'at_79", 'Quran_Abasa_80',
       'Quran_Al-Insan_76', 'Quran_Al-Ahzab_33', 'Quran_Al-Inshiqaq_84',
       'Quran_Al-Hajj_22', 'Quran_Ghafir_40', 'Quran_Ash-Sharh_94',
       "Quran_Al-An'am_6", 'Quran_Al-Mujadila_58',
       'Quran_Adh-Dhariyat_51', 'Quran_Al-Hujurat_49',
       'Quran_Az-Zukhruf_43', 'Quran_Al-Jinn_72', 'Quran_Al-Mursalat_77',
       'Quran_Al-Ahqaf_46', 'Quran_Al-Haqqah_69', 'Quran_Al-Furqan_25',
       'Quran_Al-Mutaffifin_83', 'Quran_Al-Baqarah_2_006',
       'Quran_Al-Baqarah_2_007', 'Quran_Al-Baqarah_2',
       'Quran_Al-Baqarah_2_016', 'Quran_Al-Baqarah_2_017',
       'Quran_Al-Baqarah_2_018', 'Quran_Al-Baqarah_2_019',
       'Quran_Al-Baqarah_2_020', 'Quran_Al-Muddaththir_74',
       'Quran_Al-Bayyinah_98', "Quran_Ash-Shu'ara_26", 'Quran_Al-Mulk_67',
       'Quran_Az-Zumar_39', 'Quran_Al-Fajr_89', 'Quran_An-Najm_53',
       'Quran_Taha_20', "Quran_Al-

In [406]:
quran_df = dataset[dataset["source"] == "Quran"]
def get_subset(doc_name):
    last_element = doc_name.split("_")[-1]
    if "Ali 'Imran" in doc_name or "Al-Baqarah" in doc_name:
        return int(last_element)
    else:
        return 1

new_docs = []
for doc in quran_df["doc_name"].unique():
    # swap paragraph_id and sent_id_in_paragraph cols
    curr_doc_df = quran_df[quran_df["doc_name"] == doc]
    curr_doc_df.loc[:, ["paragraph_id", "sent_id_in_paragraph"]] = curr_doc_df.loc[:, ["sent_id_in_paragraph", "paragraph_id"]].values
    curr_doc_df["subset"] = curr_doc_df["doc_name"].apply(get_subset)
    curr_doc_df["id"] = (curr_doc_df["sent_id_in_paragraph"]) + (curr_doc_df["paragraph_id"]*1000) + (curr_doc_df["subset"]*1000000) + (305*1000000000)
    new_docs.append(curr_doc_df)
new_docs_quran = pd.concat(new_docs)
# drop subset col
new_docs_quran = new_docs_quran.drop(columns=["subset"])
# reset index
new_docs_quran = new_docs_quran.reset_index(drop=True)

new_docs_quran.head()
new_docs_quran.to_csv("quran_new_docs.tsv", sep="\t")
# quran_df[quran_df["subset"] != 1]

/var/folders/5w/96k6rn392f14shpkhv9_8csh0000gq/T/ipykernel_38213/3204983304.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  curr_doc_df["subset"] = curr_doc_df["doc_name"].apply(get_subset)
/var/folders/5w/96k6rn392f14shpkhv9_8csh0000gq/T/ipykernel_38213/3204983304.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  curr_doc_df["id"] = (curr_doc_df["sent_id_in_paragraph"]) + (curr_doc_df["paragraph_id"]*1000) + (curr_doc_df["subset"]*1000000) + (305*1000000000)
/var/folders/5w/96k6rn392f14shpkhv9_8csh

In [ ]:
dataset

,id,text,source,doc_name,split,paragraph_id,sent_id_in_paragraph,word_count,new_split
0,601006001001,كلمة عربية تعني الطوفان :,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,1,5,train
1,601006001002,( أ ) الهاريكين,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,2,4,train
2,601006001003,( ب ) التورنادو,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,3,4,train
3,601006001004,( ج ) التيفون,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,4,4,train
4,601006001005,( د ) النينو,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,5,4,train


In [413]:
def adjust_id(id):
    id_str = str(id)
    if len(id_str) == 12:
        return int(id)
    else:
        subset_id = 1
        doc_id = int(id_str[:3])
        para_id = int(id_str[3:6])
        sent_id = int(id_str[6:])
        return (sent_id) + (para_id*1000) + (subset_id*1000000) + (doc_id*1000000000)

new_dataset["id"] = new_dataset["id"].apply(adjust_id)
new_dataset.head()
# new_dataset.to_csv("adjusted_splits_quran.tsv", sep="\t")


,id,text,source,doc_name,split,paragraph_id,sent_id_in_paragraph,word_count,new_split,id_digit_count
0,601006001001,كلمة عربية تعني الطوفان :,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,1,5,train,12
1,601006001002,( أ ) الهاريكين,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,2,4,train,12
2,601006001003,( ب ) التورنادو,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,3,4,train,12
3,601006001004,( ج ) التيفون,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,4,4,train,12
4,601006001005,( د ) النينو,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,5,4,train,12


In [415]:
new_dataset = new_dataset.drop(columns=["id_digit_count"])
new_dataset = new_dataset.reset_index(drop=True)
new_dataset.to_csv("adjusted_splits_quran.tsv", sep="\t")


In [407]:
# adjust dataset now with new quran docs
dataset_no_quran = dataset[dataset["source"] != "Quran"]
new_dataset = pd.concat([dataset_no_quran, new_docs_quran])
# reset index
new_dataset = new_dataset.reset_index(drop=True)
# save new dataset
new_dataset.to_csv("adjusted_splits_quran.tsv", sep="\t")
# adjust splits


In [488]:
new_dataset.head()

,id,text,source,doc_name,split,paragraph_id,sent_id_in_paragraph,word_count,new_split
0,601006001001,كلمة عربية تعني الطوفان :,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,1,5,train
1,601006001002,( أ ) الهاريكين,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,2,4,train
2,601006001003,( ب ) التورنادو,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,3,4,train
3,601006001004,( ج ) التيفون,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,4,4,train
4,601006001005,( د ) النينو,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,5,4,train


# Diwan

In [434]:
# sample 50k words from diwan
diwan_df = dataset[dataset["source"] == "Diwan"]
# get doc_names with more than 2 paragraphs

# diwan_df = diwan_df[(diwan_df["paragraph_id"] > 3) & (diwan_df["paragraph_id"] <= 30)]
max_bayt_cnt = diwan_df.groupby("doc_name")["paragraph_id"].agg("max").sort_values(ascending=False)

# filter diwan df by >3 and <= 30 abyat
diwan_df = diwan_df[diwan_df["doc_name"].isin(max_bayt_cnt[max_bayt_cnt > 3].index)]
diwan_df = diwan_df[diwan_df["doc_name"].isin(max_bayt_cnt[max_bayt_cnt <= 30].index)]
word_cnt_by_doc = diwan_df.groupby("doc_name")["word_count"].sum().sort_values(ascending=False)

In [441]:
# add word_count to each doc in diwan_df by doc_name
diwan_df["doc_word_count"] = diwan_df["doc_name"].map(word_cnt_by_doc)
# get the era by splitting doc_name by _ and taking the first element from word_cnt_by_doc

In [446]:
diwan_df["era"] = diwan_df["doc_name"].str.split("_").str[0]
doc_diwan_df = diwan_df[["doc_name", "era", "doc_word_count", "new_split"]]
doc_diwan_df = doc_diwan_df.drop_duplicates(subset=["doc_name"])
doc_diwan_df = doc_diwan_df.reset_index(drop=True)
doc_diwan_df.head()
# get the era by splitting doc_name by _ and taking the first element from word_cnt_by_doc

,doc_name,era,doc_word_count,new_split
0,العصر-الايوبي_العماد-الأصبهاني_أحاط-بورد-وجنته...,العصر-الايوبي,83,dev
1,العصر-العباسي_إبراهيم-بن-المهدي_يا-أمير-المؤمن...,العصر-العباسي,73,train
2,المخضرمون_عمرو-بن-معد-يكرب_ألم-خيال-من-أميمة-م...,المخضرمون,56,train
3,العصر-المملوكي_لسان-الدين-بن-الخطيب_أبا-مالك-أ...,العصر-المملوكي,80,train
4,العصر-الاموي_عبيد-الله-بن-الرقيات_هل-تعرف-الرب...,العصر-الاموي,54,train


In [474]:
# sample greedily by word count to match needed using era_dist_wc dict
# import pandas as pd

def greedy_sample_docs_by_era(
    df,
    era_dist_wc,
    era_col="era",
    doc_col="doc_name",
    wc_col="doc_word_count",
    random_state=None,
    largest_first=True,
):
    """
    Greedily sample whole documents for each era to get close to target word counts.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain one row per document, with columns:
        [doc_name, era, doc_word_count]
    era_dist_wc : dict
        Mapping: era -> target word count
    era_col : str
        Name of era column
    doc_col : str
        Name of document name/id column
    wc_col : str
        Name of document word count column
    random_state : int or None
        For tie-breaking shuffle if needed
    largest_first : bool
        If True, sort docs by descending word count before greedy selection.
        If False, ascending.

    Returns
    -------
    selected_docs : dict
        {era: [doc_name1, doc_name2, ...]}
    summary_df : pd.DataFrame
        Per-era summary with target, achieved, difference, num_docs
    """
    work_df = df[[doc_col, era_col, wc_col]].drop_duplicates(subset=[doc_col]).copy()

    if random_state is not None:
        work_df = work_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    selected_docs = {}
    summary_rows = []

    for era, target_wc in era_dist_wc.items():
        era_df = work_df[work_df[era_col] == era].copy()

        if era_df.empty:
            selected_docs[era] = []
            summary_rows.append({
                "era": era,
                "target_wc": target_wc,
                "achieved_wc": 0,
                "difference": -target_wc,
                "num_docs": 0,
            })
            continue

        era_df = era_df.sort_values(
            by=wc_col,
            ascending=not largest_first
        ).reset_index(drop=True)

        chosen = []
        total = 0

        for _, row in era_df.iterrows():
            doc_wc = row[wc_col]

            # If we haven't reached the target yet, consider taking the doc
            if total < target_wc:
                without_doc_diff = abs(target_wc - total)
                with_doc_diff = abs(target_wc - (total + doc_wc))

                # take it if it improves closeness OR we are still below target
                if with_doc_diff <= without_doc_diff or total < target_wc:
                    chosen.append(row[doc_col])
                    total += doc_wc

                    # optional early stop:
                    # if adding more would only make things worse, we stop later
            else:
                break

        # Final local improvement:
        # compare current set vs removing the last added doc
        if chosen:
            last_doc = chosen[-1]
            last_wc = era_df.loc[era_df[doc_col] == last_doc, wc_col].iloc[0]

            if abs(target_wc - (total - last_wc)) < abs(target_wc - total):
                chosen = chosen[:-1]
                total -= last_wc

        selected_docs[era] = chosen
        summary_rows.append({
            "era": era,
            "target_wc": target_wc,
            "achieved_wc": total,
            "difference": total - target_wc,
            "num_docs": len(chosen),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("era").reset_index(drop=True)
    return selected_docs, summary_df

def greedy_wc_split_by_era(
    df,
    era_col="era",
    doc_col="doc_name",
    wc_col="doc_word_count",
    split_ratios=None,
    random_state=42,
):
    """
    Split whole documents into train/dev/test so that, within each era,
    the WORD COUNTS are as close as possible to the requested ratios.

    Returns
    -------
    doc_split_map : dict
        {doc_name: split}
    doc_level_df : pd.DataFrame
        One row per doc with assigned split
    summary_df : pd.DataFrame
        Per-era target vs achieved word counts
    """
    if split_ratios is None:
        split_ratios = {"train": 0.70, "dev": 0.15, "test": 0.15}

    # one row per document
    doc_df = (
        df[[doc_col, era_col, wc_col]]
        .drop_duplicates(subset=[doc_col])
        .copy()
    )

    # shuffle first so ties are not always resolved the same way
    doc_df = doc_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    assigned_docs = []
    summary_rows = []

    for era, era_df in doc_df.groupby(era_col, sort=False):
        era_df = era_df.copy()

        total_wc = era_df[wc_col].sum()
        targets = {split: total_wc * ratio for split, ratio in split_ratios.items()}
        current = {split: 0 for split in split_ratios}

        # sort biggest docs first
        era_df = era_df.sort_values(wc_col, ascending=False).reset_index(drop=True)

        for _, row in era_df.iterrows():
            doc_wc = row[wc_col]

            best_split = None
            best_score = None

            # try placing this doc in each split, choose the placement
            # that minimizes total absolute error to word-count targets
            for split in split_ratios:
                trial = current.copy()
                trial[split] += doc_wc

                score = sum(abs(trial[s] - targets[s]) for s in split_ratios)

                if best_score is None or score < best_score:
                    best_score = score
                    best_split = split

            current[best_split] += doc_wc
            assigned_docs.append((row[doc_col], era, row[wc_col], best_split))

        for split in split_ratios:
            summary_rows.append({
                "era": era,
                "split": split,
                "target_wc": targets[split],
                "achieved_wc": current[split],
                "difference": current[split] - targets[split],
            })

    doc_level_df = pd.DataFrame(
        assigned_docs,
        columns=[doc_col, era_col, wc_col, "wc_split"]
    )

    doc_split_map = dict(zip(doc_level_df[doc_col], doc_level_df["wc_split"]))
    summary_df = pd.DataFrame(summary_rows)

    return doc_split_map, doc_level_df, summary_df

selected_docs, summary_df = greedy_sample_docs_by_era(
    df=doc_diwan_df,
    era_dist_wc=era_dist_wc,
    era_col="era",
    doc_col="doc_name",
    wc_col="doc_word_count",
    random_state=42,
    largest_first=False,
)

selected_doc_set = {
    doc
    for docs_in_era in selected_docs.values()
    for doc in docs_in_era
}
selected_doc_set.add('العصر-الاسلامي_فاطمة-الزهراء_أمرك-يا-ابن-عم-سمع-وطاعه_001')

# add keep column
doc_diwan_df["keep_doc"] = doc_diwan_df["doc_name"].isin(selected_doc_set)
# doc_diwan_df.head()
# # add keep column
# doc_diwan_df["keep_doc"] = doc_diwan_df["doc_name"].isin(selected_doc_set)
sampled_diwan_df = doc_diwan_df[doc_diwan_df["keep_doc"] == True].copy()
doc_split_map_splits, _, summary_df_splits = greedy_wc_split_by_era(
    sampled_diwan_df,
    era_col="era",
    doc_col="doc_name",
    wc_col="doc_word_count",
    split_ratios={"train": 0.70, "dev": 0.15, "test": 0.15},
    random_state=42
)

In [481]:
sampled_diwan_df["split"] = sampled_diwan_df["doc_name"].map(doc_split_map_splits)
sampled_diwan_df.to_csv("sampled_diwan_df.tsv", sep="\t")

In [475]:
wc_table = summary_df_splits.pivot(
    index="era",
    columns="split",
    values="achieved_wc"
).fillna(0)

wc_table

split,dev,test,train
era,,,
العصر-الأندلسي,705,704,3323
العصر-الاسلامي,29,28,99
العصر-الاموي,690,672,3176
العصر-الايوبي,941,925,4372
العصر-الجاهلي,246,264,1207
العصر-العباسي,2801,2793,13084
العصر-العثماني,777,789,3652
العصر-المملوكي,987,985,4626
المخضرمون,321,318,1505


In [ ]:
doc_split_map, summary_df, doc_level_df = split_docs_by_era_wordcount(
    diwan_df,
    era_col="era",
    doc_col="doc_name",
    wc_col="doc_word_count",

    split_ratios={"train": 0.70, "dev": 0.15, "test": 0.15},
    random_state=42,
    largest_first=False,
)

In [459]:
summary_df

,era,split,target_wc,achieved_wc,difference
0,العصر-الاموي,train,209764.80,209765,0.20
1,العصر-الاموي,dev,44949.60,44940,-9.60
2,العصر-الاموي,test,44949.60,44959,9.40
3,العصر-الجاهلي,train,79429.70,79434,4.30
4,العصر-الجاهلي,dev,17020.65,17014,-6.65
5,العصر-الجاهلي,test,17020.65,17023,2.35
6,العصر-العباسي,train,734211.10,734211,-0.10
7,العصر-العباسي,dev,157330.95,157323,-7.95
8,العصر-العباسي,test,157330.95,157339,8.05
9,العصر-العثماني,train,287237.30,287237,-0.30


In [452]:
era_dist_wc = {}
for era in era_dist.keys():
    era_dist_wc[era] = int(round(era_dist[era] * 50000) / 100.)
    print(f"{era}: {era_dist_wc[era]}")


العصر-العباسي: 18686
العصر-المملوكي: 6584
العصر-الايوبي: 6245
العصر-العثماني: 5226
العصر-الأندلسي: 4742
العصر-الاموي: 4548
المخضرمون: 2143
العصر-الجاهلي: 1716
العصر-الاسلامي: 106


In [440]:
# sample 50k words according to era distribution
era_dist = word_cnt_by_doc.index.str.split("_").str[0].value_counts(normalize=True)
era_dist = era_dist.to_dict()
era_dist

# get needed word count for each era 
for era in era_dist.keys():
    era_word_cnt = int(era_dist[era] * 50000)
    print(f"{era}: {era_word_cnt}")

# get needed docs for each era



العصر-العباسي: 18829
العصر-المملوكي: 6913
العصر-الايوبي: 6317
العصر-العثماني: 5005
العصر-الاموي: 4569
العصر-الأندلسي: 4523
المخضرمون: 2001
العصر-الجاهلي: 1646
العصر-الاسلامي: 194


In [513]:
new_dataset_sampled_diwan = new_dataset.copy()
print(len(new_dataset_sampled_diwan[new_dataset_sampled_diwan["source"] == "Diwan"]["doc_name"].unique()))
# keep only docs in sampled_diwan_df
new_dataset_sampled_diwan = new_dataset_sampled_diwan[new_dataset_sampled_diwan["doc_name"].isin(sampled_diwan_df["doc_name"])]
# map splits to new_split column
new_dataset_sampled_diwan["new_split"] = new_dataset_sampled_diwan["doc_name"].map(doc_split_map_splits)
print(len(new_dataset_sampled_diwan[new_dataset_sampled_diwan["source"] == "Diwan"]["doc_name"].unique()))
# reset index and save
non_diwan_df = new_dataset[new_dataset["source"] != "Diwan"]
new_dataset_sampled_diwan = new_dataset_sampled_diwan.reset_index(drop=True)
new_dataset_sampled_diwan = pd.concat([new_dataset_sampled_diwan, non_diwan_df])
new_dataset_sampled_diwan = new_dataset_sampled_diwan.reset_index(drop=True)
new_dataset_sampled_diwan.to_csv("3_new_dataset_sampled_diwan+quran.tsv", sep="\t")

73966
1683


# Subtitles + ChatGPT

In [492]:
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.charmap import CharMapper
from camel_tools.utils.transliterate import Transliterator
from camel_tools.tokenizers.word import simple_word_tokenize

def clean(text):
    arclean = CharMapper.mapper_from_json("./arclean_map.json")
    arclean_translit = Transliterator(arclean)
    out = arclean_translit.transliterate(text)
    out = re.sub(r'(?<=\B)ى(?=\B)', 'ي', out)
    return out

# clean using clean -> simple word tokenize, split_digits=True -> dediac_ar
def clean_text(text, tokenize=False):
    text = clean(text)
    text = simple_word_tokenize(text, split_digits=True)
    text = [dediac_ar(word) for word in text]
    if not tokenize:
        return " ".join(text)
    return text

In [485]:
# read subtitles
barec_annotations = pd.read_excel("../../../Seg/ReconstructingDocs/BAREC_1M_v1_annotations (1).xlsx", sheet_name="all sents sorted")

In [502]:
subtitles_df.head()

,ID,Original Sent,Mask Sent,File,Subset,File+subset,Document ID,Sent ID,Sentence ID,Split,...,annotation file,New Readability Level,New RL_num,New Annotator,New annotation file,Project Phase,Source.1,Release Filter,ALC filter on ReadME,Source.2
65659,65660,أريدها أن ترحل,أريدها أن ترحل,BAREC_Subtitles_Subs.txt,1,BAREC_Subtitles_Subs_001.txt,2100001,1,21000010001,Train,...,Sawsan_phase1_14.xlsx_revised,10-ya,10,Sawsan,Sawsan_phase1_14.xlsx_revised,Phase1,Subtitles,Yes,NaN,Subtitles
65660,65661,- فلنتخلص منها,فلنتخلص منها,BAREC_Subtitles_Subs.txt,1,BAREC_Subtitles_Subs_001.txt,2100001,2,21000010002,Train,...,Zeina_phase1_23.xlsx_revised,8-Ha,8,Zeina,Zeina_phase1_23.xlsx_revised,Phase1,Subtitles,Yes,NaN,Subtitles
65661,65662,- لقد رأيناه من قبل .,لقد رأيناه من قبل,BAREC_Subtitles_Subs.txt,1,BAREC_Subtitles_Subs_001.txt,2100001,3,21000010003,Train,...,Zeina_phase1_13.xlsx_revised,10-ya,10,Zeina,Zeina_phase1_13.xlsx_revised,Phase1,Subtitles,Yes,NaN,Subtitles
65662,65663,أعرف ان مافعله زوجي فظيع أعرف .,أعرف ان مافعله زوجي فظيع أعرف,BAREC_Subtitles_Subs.txt,1,BAREC_Subtitles_Subs_001.txt,2100001,4,21000010004,Train,...,Mirvat_phase1_22.xlsx_revised,10-ya,10,Mirvat,Mirvat_phase1_22.xlsx_revised,Phase1,Subtitles,Yes,NaN,Subtitles
65663,65664,لكن الآن أنا في الواقع أركض خلف,لكن الآن أنا في الواقع أركض خلف,BAREC_Subtitles_Subs.txt,1,BAREC_Subtitles_Subs_001.txt,2100001,5,21000010005,Train,...,###,###,###,###,###,###,###,Yes,NaN,Subtitles


In [507]:
subtitles_df = barec_annotations[barec_annotations["Source"] == "Subtitles"]

# group by doc and generate the needed columns
subs_df = []
for doc in subtitles_df["File+subset"].unique():
    curr_doc_df = subtitles_df[subtitles_df["File+subset"] == doc]
    # sort by Sentence ID column
    curr_doc_df = curr_doc_df.sort_values(by="Sentence ID")
    # get the doc name by splitting the doc name by + and taking the first element
    doc_name = doc[:-4]
    # get the subset by splitting the doc name by + and taking the second element
    subset = int(doc_name[-3:])
    paragraph_id = 1
    curr_doc_new_df_rows = []
    doc_split = curr_doc_df["Split"].iloc[0].lower()
    for _, row in curr_doc_df.iterrows():
        cleaned_text = clean_text(row["Original Sent"])
        
        ex = {
            "id": 602*1_000_000_000 +  subset*1_000_000 + paragraph_id*1_000 + 1,
            "text": cleaned_text,
            "source": "Subtitles",
            "doc_name": doc_name,
            "split": doc_split,
            "paragraph_id": paragraph_id,
            "sent_id_in_paragraph": 1,
            "word_count": len(cleaned_text.split(" ")),
            "new_split": doc_split,
        }
        paragraph_id += 1
        curr_doc_new_df_rows.append(ex)
    curr_doc_new_df = pd.DataFrame(curr_doc_new_df_rows)
    subs_df.append(curr_doc_new_df)
subs_df = pd.concat(subs_df)
# subs_df.to_csv("subs_df.tsv", sep="\t")
subs_df = subs_df.reset_index(drop=True)
subs_df.head()



,id,text,source,doc_name,split,paragraph_id,sent_id_in_paragraph,word_count,new_split
0,602001001001,أريدها أن ترحل,Subtitles,BAREC_Subtitles_Subs_001,train,1,1,3,train
1,602001002001,- فلنتخلص منها,Subtitles,BAREC_Subtitles_Subs_001,train,2,1,3,train
2,602001003001,- لقد رأيناه من قبل .,Subtitles,BAREC_Subtitles_Subs_001,train,3,1,6,train
3,602001004001,أعرف ان مافعله زوجي فظيع أعرف .,Subtitles,BAREC_Subtitles_Subs_001,train,4,1,7,train
4,602001005001,لكن الآن أنا في الواقع أركض خلف,Subtitles,BAREC_Subtitles_Subs_001,train,5,1,7,train


In [ ]:
chatgpt_df = barec_annotations[barec_annotations["Source"] == "chatGPT"]

# group by doc and generate the needed columns
gpt_df = []
for doc in chatgpt_df["File+subset"].unique():
    curr_doc_df = chatgpt_df[chatgpt_df["File+subset"] == doc]
    # sort by Sentence ID column
    curr_doc_df = curr_doc_df.sort_values(by="Sentence ID")
    # get the doc name by splitting the doc name by + and taking the first element
    doc_name = doc[:-4]
    # get the subset by splitting the doc name by + and taking the second element
    subset = int(doc_name[-3:])
    paragraph_id = 1
    curr_doc_new_df_rows = []
    doc_split = curr_doc_df["Split"].iloc[0].lower()
    for _, row in curr_doc_df.iterrows():
        cleaned_text = clean_text(row["Original Sent"])
        
        ex = {
            "id": 603*1_000_000_000 +  subset*1_000_000 + paragraph_id*1_000 + 1,
            "text": cleaned_text,
            "source": "chatGPT",
            "doc_name": doc_name,
            "split": doc_split,
            "paragraph_id": paragraph_id,
            "sent_id_in_paragraph": 1,
            "word_count": len(cleaned_text.split(" ")),
            "new_split": doc_split,
        }
        paragraph_id += 1
        curr_doc_new_df_rows.append(ex)
    curr_doc_new_df = pd.DataFrame(curr_doc_new_df_rows)
    gpt_df.append(curr_doc_new_df)
gpt_df = pd.concat(gpt_df)
# subs_df.to_csv("subs_df.tsv", sep="\t")
gpt_df = gpt_df.reset_index(drop=True)
gpt_df.head()

,id,text,source,doc_name,split,paragraph_id,sent_id_in_paragraph,word_count,new_split
0,603001001001,الكتاب على الطاولة والزهرة في الحديقة .,chatGPT,BAREC_chatGPT_3.5_100_sents_8to10_words_001,dev,1,1,7,dev
1,603001002001,الأرنب يقفز فوق العشب الأخضر في الصباح الباكر .,chatGPT,BAREC_chatGPT_3.5_100_sents_8to10_words_001,dev,2,1,9,dev
2,603001003001,الأميرة تلبس فستانا جميلا وتحمل وردة حمراء .,chatGPT,BAREC_chatGPT_3.5_100_sents_8to10_words_001,dev,3,1,8,dev
3,603001004001,الأطفال يلعبون في الملعب ويضحكون بسعادة كبيرة .,chatGPT,BAREC_chatGPT_3.5_100_sents_8to10_words_001,dev,4,1,8,dev
4,603001005001,الشمس تشرق في السماء وتضيء العالم بألوانها الد...,chatGPT,BAREC_chatGPT_3.5_100_sents_8to10_words_001,dev,5,1,9,dev


In [516]:
# add gpt_df and subs_df to the dataset
new_dataset_sampled_diwan_subs_gpt = pd.concat([new_dataset_sampled_diwan, gpt_df, subs_df])
new_dataset_sampled_diwan_subs_gpt = new_dataset_sampled_diwan_subs_gpt.reset_index(drop=True)
new_dataset_sampled_diwan_subs_gpt.to_csv("4_new_dataset_sampled_diwan_subs_gpt.tsv", sep="\t", index=False)
new_dataset_sampled_diwan_subs_gpt.head()


,id,text,source,doc_name,split,paragraph_id,sent_id_in_paragraph,word_count,new_split
0,701001001001,وإذا تضايق باب رز <SEG> قك فادرع ثوب اصطبارك,Diwan,العصر-العباسي_الأحنف-العكبري_وإذا-تضايق-باب-رز...,No-Split,1,1,8,train
1,701001002001,والدهر في حركاته <SEG> وسكونه كل اعتبارك,Diwan,العصر-العباسي_الأحنف-العكبري_وإذا-تضايق-باب-رز...,No-Split,2,1,6,train
2,701001003001,لا تعتبن على الزما <SEG> ن فليس يجري باختيارك,Diwan,العصر-العباسي_الأحنف-العكبري_وإذا-تضايق-باب-رز...,No-Split,3,1,8,train
3,701001004001,فلرب ذنب قد أبر <SEG> على انتخابك واختصارك,Diwan,العصر-العباسي_الأحنف-العكبري_وإذا-تضايق-باب-رز...,No-Split,4,1,7,train
4,701001001001,لقد أذاب الكبدا <SEG> هوى وأضنى الجسدا,Diwan,العصر-العباسي_خالد-الكاتب_لقد-أذاب-الكبدا_001,No-Split,1,1,6,test


# Stats

In [349]:
pnx_mapping = {
    '.': '.',
    '?': '?',
    ';': ';',
    ',': '،'
}

In [350]:
dataset.head()

,id,text,source,doc_name,split,paragraph_id,sent_id_in_paragraph,word_count,new_split
0,601006001001,كلمة عربية تعني الطوفان :,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,1,5,train
1,601006001002,( أ ) الهاريكين,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,2,4,train
2,601006001003,( ب ) التورنادو,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,3,4,train
3,601006001004,( ج ) التيفون,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,4,4,train
4,601006001005,( د ) النينو,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,5,4,train


In [351]:
dataset.columns

Index(['id', 'text', 'source', 'doc_name', 'split', 'paragraph_id',
       'sent_id_in_paragraph', 'word_count', 'new_split'],
      dtype='object')

In [356]:
from collections import defaultdict
SENT_END_PUNCT = [ ".", ",", "،", ";", "؛", "!", "?", "؟"]

def has_sent_end_punct(text):
    return text.split(" ")[-1] in SENT_END_PUNCT

def read_dataset(df):
    data = defaultdict(list)
    for split in df["new_split"].unique():
        split_df = df[df["new_split"] == split]
        for _, row in split_df.iterrows():
            ex = {
                'id': row['id'],
                'sent': row['text'],
                'source': row['source'],
                'document': row['doc_name'],
                'length': row['word_count'],
                'split': split,
                'has_sent_end_punct': has_sent_end_punct(row['text']),
                'labels': [0] * (len(row["text"].split(" ")) - 1) + [1]
            }
            data[split].append(ex)
    return data

def read_barec(dataset, remove_all_puncts=False):
    data = defaultdict(list)
    for split in dataset.keys():
        print(f"Processing split: {split} with {len(dataset[split])} examples")
        for example in dataset[split]:

            ex = {
                'split': split,
                'id': example['ID'], 
                'sent': example['Word'], 
                'domain': example['Source'],
                'document': example['Document'], 
                'length': len(example['Word'].split(" ")), 
                "Annotator": example['Annotator'], 
                "R19": example["Readability_Level_19"],
                "has_sent_end_punct": has_sent_end_punct(example['Word']),
                }
            if remove_all_puncts and split in ["train"]:
                
                ex["sent"] = re.sub(r'[^\w\s]', '', ex["sent"])
                ex["sent"] = re.sub(r'\s+', ' ', ex["sent"]).strip()
                ex["length"] = len(ex["sent"].split(" "))
                ex["has_sent_end_punct"] = has_sent_end_punct(ex["sent"])
                
            ex["labels"] = [0] * (ex["length"] - 1) + [1] # if (not ex["has_sent_end_punct"]) else [0] * ex["length"]
            assert len(ex["labels"]) == ex["length"], f"Length of labels {len(ex['labels'])} does not match length of text {ex['length']}"
            data[split].append(ex) #"num_sent_end_punct": cnt_sent_end_punct(example['Word']), "has_sent_end_punct": has_sent_end_punct(example['Word']), "split": split, "ends_with_sent_end_punct": ends_with_sent_end_punct(example['Word'])})
            
    return data

# data = read_barec(dataset, remove_all_puncts=False)
data = read_dataset(dataset)

In [ ]:
# train_data = data["train"]

In [357]:
grouped_examples_by_split = defaultdict(list)
for split in data:
    for example in data[split]:
        if example['split'] == split:
            grouped_examples_by_split[split].append(example)
grouped_examples_by_doc = defaultdict(lambda: defaultdict(list))
for split in data:
    for example in data[split]:
        if example['split'] == split:
            grouped_examples_by_doc[split][example['document']].append(example)

In [358]:
def generate_grouped_doc_data(data):
    grouped_doc_data = defaultdict(lambda: defaultdict(list))
    for split in data:
        for doc in data[split]:
            # sort examples in doc by ID
            # print(data[split][doc])
            data[split][doc] = sorted(data[split][doc], key=lambda x: x['id'])
            curr_doc_text = []
            curr_doc_labels = []
            for i in data[split][doc]:
                curr_doc_text.extend(i['sent'].split(" "))
                curr_doc_labels.extend(i['labels'])
            assert len(curr_doc_labels) == len(curr_doc_text), f"Length of labels {len(curr_doc_labels)} does not match length of text {curr_doc_text}, source: {data[split][doc][0]['source']}"
            curr_doc_source = data[split][doc][0]['source']
            curr_doc_lengths = [i['length'] for i in data[split][doc]]
            # curr_doc_r19 = [i['R19'] for i in data[split][doc]]
            grouped_doc_data[split][doc] = {"text": curr_doc_text,
            "labels": curr_doc_labels,
            "source": curr_doc_source, 
            "lengths": curr_doc_lengths,  
            "document": doc,
            "source": data[split][doc][0]['source'], 
            "joined_text": " ".join(curr_doc_text)
            }
    return grouped_doc_data

grouped_doc_data = generate_grouped_doc_data(grouped_examples_by_doc)

In [24]:
import string
from collections import Counter

def count_consecutive_punctuations(text, delimiter=' ', pnx_set=None, same_type_only=True, labels=None):
    """
    Count consecutive punctuation sequences in tokenized text.
    
    Args:
        text: Space-separated tokenized text
        delimiter: Token separator (default: ' ')
        pnx_set: Custom punctuation set (default: string.punctuation)
        same_type_only: Only count sequences of same punctuation type
        labels: Optional list of labels (same length as tokens) to track sentence boundaries
    
    Returns:
        dict with keys:
            - 'counts': dict mapping sequences to their frequency
            - 'with_boundary': dict mapping sequences to count of those containing a 1 label
            - 'without_boundary': dict mapping sequences to count of those without any 1 label
            - 'collapsed_counts': dict mapping collapsed patterns (e.g., '.*') to their frequency
            - 'collapsed_with_boundary': dict mapping collapsed patterns to count with boundaries
            - 'collapsed_without_boundary': dict mapping collapsed patterns to count without boundaries
    """
    if pnx_set is None:
        punctuation_set = set(string.punctuation)
    else:
        punctuation_set = pnx_set
    
    tokens = text.split(delimiter)
    
    consecutive_sequences = []
    sequence_has_boundary = []  # Track if each sequence has a 1 in its labels
    current_sequence = []
    current_indices = []
    
    for idx, token in enumerate(tokens):
        # Check if token is punctuation (all characters are punctuation)
        if token and all(char in punctuation_set for char in token):
            current_sequence.append(token)
            current_indices.append(idx)
        else:
            # End of punctuation sequence
            if len(current_sequence) > 1:  # Only count if 2+ consecutive punctuations
                should_add = True
                if same_type_only:
                    should_add = len(set(current_sequence)) == 1
                
                if should_add:
                    consecutive_sequences.append(' '.join(current_sequence))
                    # Check if any label in this sequence is 1
                    if labels is not None:
                        has_boundary = any(labels[i] == 1 for i in current_indices)
                        sequence_has_boundary.append(has_boundary)
                    else:
                        sequence_has_boundary.append(None)
            current_sequence = []
            current_indices = []
    
    # Don't forget the last sequence
    if len(current_sequence) > 1:
        should_add = True
        if same_type_only:
            should_add = len(set(current_sequence)) == 1
        
        if should_add:
            consecutive_sequences.append(' '.join(current_sequence))
            if labels is not None:
                has_boundary = any(labels[i] == 1 for i in current_indices)
                sequence_has_boundary.append(has_boundary)
            else:
                sequence_has_boundary.append(None)

    # Build result counts
    counts = Counter([f"<s>{pnxs}<s>" for pnxs in consecutive_sequences])
    
    # Count sequences with and without boundaries
    with_boundary = Counter()
    without_boundary = Counter()
    
    for seq, has_bound in zip(consecutive_sequences, sequence_has_boundary):
        key = f"<s>{seq}<s>"
        if has_bound is True:
            with_boundary[key] += 1
        elif has_bound is False:
            without_boundary[key] += 1
    
    # Build collapsed counts (e.g., ". ." and ". . ." both become ".*")
    def get_collapsed_key(seq):
        """Convert a sequence like '. .' or '. . .' to collapsed form like '.*'"""
        # Get the first token (they're all the same since same_type_only=True)
        first_token = seq.split()[0]
        return f"{first_token}*"
    
    collapsed_counts = Counter()
    collapsed_with_boundary = Counter()
    collapsed_without_boundary = Counter()
    
    for seq, has_bound in zip(consecutive_sequences, sequence_has_boundary):
        collapsed_key = get_collapsed_key(seq)
        collapsed_counts[collapsed_key] += 1
        if has_bound is True:
            collapsed_with_boundary[collapsed_key] += 1
        elif has_bound is False:
            collapsed_without_boundary[collapsed_key] += 1
    
    return {
        'counts': dict(counts),
        'with_boundary': dict(with_boundary),
        'without_boundary': dict(without_boundary),
        'collapsed_counts': dict(collapsed_counts),
        'collapsed_with_boundary': dict(collapsed_with_boundary),
        'collapsed_without_boundary': dict(collapsed_without_boundary)
    }

# Initialize counters for counts, with_boundary, and without_boundary
consecutive_pnx_counts_train = defaultdict(lambda: defaultdict(int))
consecutive_pnx_counts_dev = defaultdict(lambda: defaultdict(int))
consecutive_pnx_counts_test = defaultdict(lambda: defaultdict(int))

consecutive_pnx_with_boundary_train = defaultdict(lambda: defaultdict(int))
consecutive_pnx_with_boundary_dev = defaultdict(lambda: defaultdict(int))
consecutive_pnx_with_boundary_test = defaultdict(lambda: defaultdict(int))

consecutive_pnx_without_boundary_train = defaultdict(lambda: defaultdict(int))
consecutive_pnx_without_boundary_dev = defaultdict(lambda: defaultdict(int))
consecutive_pnx_without_boundary_test = defaultdict(lambda: defaultdict(int))

# Initialize counters for collapsed counts
collapsed_pnx_counts_train = defaultdict(lambda: defaultdict(int))
collapsed_pnx_counts_dev = defaultdict(lambda: defaultdict(int))
collapsed_pnx_counts_test = defaultdict(lambda: defaultdict(int))

collapsed_pnx_with_boundary_train = defaultdict(lambda: defaultdict(int))
collapsed_pnx_with_boundary_dev = defaultdict(lambda: defaultdict(int))
collapsed_pnx_with_boundary_test = defaultdict(lambda: defaultdict(int))

collapsed_pnx_without_boundary_train = defaultdict(lambda: defaultdict(int))
collapsed_pnx_without_boundary_dev = defaultdict(lambda: defaultdict(int))
collapsed_pnx_without_boundary_test = defaultdict(lambda: defaultdict(int))

for split in grouped_doc_data:
    for doc in grouped_doc_data[split]:
        text_to_parse = grouped_doc_data[split][doc]["joined_text"]
        labels = grouped_doc_data[split][doc]["labels"]
        source = grouped_doc_data[split][doc]["domain"]
        output = count_consecutive_punctuations(text_to_parse, labels=labels)
        
        for key, value in output['counts'].items():
            if split == "train":
                consecutive_pnx_counts_train[source][key] += value
            elif split == "dev":
                consecutive_pnx_counts_dev[source][key] += value
            elif split == "test":
                consecutive_pnx_counts_test[source][key] += value
        
        for key, value in output['with_boundary'].items():
            if split == "train":
                consecutive_pnx_with_boundary_train[source][key] += value
            elif split == "dev":
                consecutive_pnx_with_boundary_dev[source][key] += value
            elif split == "test":
                consecutive_pnx_with_boundary_test[source][key] += value
        
        for key, value in output['without_boundary'].items():
            if split == "train":
                consecutive_pnx_without_boundary_train[source][key] += value
            elif split == "dev":
                consecutive_pnx_without_boundary_dev[source][key] += value
            elif split == "test":
                consecutive_pnx_without_boundary_test[source][key] += value
        
        # Process collapsed counts
        for key, value in output['collapsed_counts'].items():
            if split == "train":
                collapsed_pnx_counts_train[source][key] += value
            elif split == "dev":
                collapsed_pnx_counts_dev[source][key] += value
            elif split == "test":
                collapsed_pnx_counts_test[source][key] += value
        
        for key, value in output['collapsed_with_boundary'].items():
            if split == "train":
                collapsed_pnx_with_boundary_train[source][key] += value
            elif split == "dev":
                collapsed_pnx_with_boundary_dev[source][key] += value
            elif split == "test":
                collapsed_pnx_with_boundary_test[source][key] += value
        
        for key, value in output['collapsed_without_boundary'].items():
            if split == "train":
                collapsed_pnx_without_boundary_train[source][key] += value
            elif split == "dev":
                collapsed_pnx_without_boundary_dev[source][key] += value
            elif split == "test":
                collapsed_pnx_without_boundary_test[source][key] += value

# Sort all counters by value descending
for source in consecutive_pnx_counts_train:
    consecutive_pnx_counts_train[source] = sorted(consecutive_pnx_counts_train[source].items(), key=lambda x: x[1], reverse=True)
for source in consecutive_pnx_counts_dev:
    consecutive_pnx_counts_dev[source] = sorted(consecutive_pnx_counts_dev[source].items(), key=lambda x: x[1], reverse=True)
for source in consecutive_pnx_counts_test:
    consecutive_pnx_counts_test[source] = sorted(consecutive_pnx_counts_test[source].items(), key=lambda x: x[1], reverse=True)

for source in consecutive_pnx_with_boundary_train:
    consecutive_pnx_with_boundary_train[source] = sorted(consecutive_pnx_with_boundary_train[source].items(), key=lambda x: x[1], reverse=True)
for source in consecutive_pnx_with_boundary_dev:
    consecutive_pnx_with_boundary_dev[source] = sorted(consecutive_pnx_with_boundary_dev[source].items(), key=lambda x: x[1], reverse=True)
for source in consecutive_pnx_with_boundary_test:
    consecutive_pnx_with_boundary_test[source] = sorted(consecutive_pnx_with_boundary_test[source].items(), key=lambda x: x[1], reverse=True)

for source in consecutive_pnx_without_boundary_train:
    consecutive_pnx_without_boundary_train[source] = sorted(consecutive_pnx_without_boundary_train[source].items(), key=lambda x: x[1], reverse=True)
for source in consecutive_pnx_without_boundary_dev:
    consecutive_pnx_without_boundary_dev[source] = sorted(consecutive_pnx_without_boundary_dev[source].items(), key=lambda x: x[1], reverse=True)
for source in consecutive_pnx_without_boundary_test:
    consecutive_pnx_without_boundary_test[source] = sorted(consecutive_pnx_without_boundary_test[source].items(), key=lambda x: x[1], reverse=True)

# Sort collapsed counters by value descending
for source in collapsed_pnx_counts_train:
    collapsed_pnx_counts_train[source] = sorted(collapsed_pnx_counts_train[source].items(), key=lambda x: x[1], reverse=True)
for source in collapsed_pnx_counts_dev:
    collapsed_pnx_counts_dev[source] = sorted(collapsed_pnx_counts_dev[source].items(), key=lambda x: x[1], reverse=True)
for source in collapsed_pnx_counts_test:
    collapsed_pnx_counts_test[source] = sorted(collapsed_pnx_counts_test[source].items(), key=lambda x: x[1], reverse=True)

for source in collapsed_pnx_with_boundary_train:
    collapsed_pnx_with_boundary_train[source] = sorted(collapsed_pnx_with_boundary_train[source].items(), key=lambda x: x[1], reverse=True)
for source in collapsed_pnx_with_boundary_dev:
    collapsed_pnx_with_boundary_dev[source] = sorted(collapsed_pnx_with_boundary_dev[source].items(), key=lambda x: x[1], reverse=True)
for source in collapsed_pnx_with_boundary_test:
    collapsed_pnx_with_boundary_test[source] = sorted(collapsed_pnx_with_boundary_test[source].items(), key=lambda x: x[1], reverse=True)

for source in collapsed_pnx_without_boundary_train:
    collapsed_pnx_without_boundary_train[source] = sorted(collapsed_pnx_without_boundary_train[source].items(), key=lambda x: x[1], reverse=True)
for source in collapsed_pnx_without_boundary_dev:
    collapsed_pnx_without_boundary_dev[source] = sorted(collapsed_pnx_without_boundary_dev[source].items(), key=lambda x: x[1], reverse=True)
for source in collapsed_pnx_without_boundary_test:
    collapsed_pnx_without_boundary_test[source] = sorted(collapsed_pnx_without_boundary_test[source].items(), key=lambda x: x[1], reverse=True)


In [25]:
consecutive_pnx_counts_train

defaultdict(<function __main__.<lambda>()>,
            {'Majed': [('<s>. .<s>', 2184),
              ('<s>! !<s>', 82),
              ('<s>. . .<s>', 60),
              ('<s>* *<s>', 32),
              ('<s>! ! !<s>', 14),
              ('<s>, , ,<s>', 3),
              ('<s>. . . .<s>', 3),
              ('<s>" "<s>', 2),
              ('<s>@ @ @<s>', 1),
              ('<s>. . . . . .<s>', 1),
              ('<s>. . . . . . .<s>', 1)],
             'ArabicMMLU': [('<s>. . .<s>', 89),
              ('<s>. . . . . . .<s>', 52),
              ('<s>. . . . . . . .<s>', 43),
              ('<s>. .<s>', 40),
              ('<s>. . . . . . . . .<s>', 30),
              ('<s>. . . . . . . . . .<s>', 28),
              ('<s>. . . . .<s>', 23),
              ('<s>. . . . . .<s>', 22),
              ('<s>. . . .<s>', 20),
              ('<s>. . . . . . . . . . .<s>', 7),
              ('<s>. . . . . . . . . . . .<s>', 7),
              ('<s>. . . . . . . . . . . . .<s>', 6),
              ('<s

In [27]:
# Convert collapsed_pnx_counts_train to DataFrame with boundary percentage
# First, convert collapsed_pnx_with_boundary_train to a dict for easy lookup
boundary_lookup = {}
for source, pnx_counts in collapsed_pnx_with_boundary_train.items():
    for pnx, count in pnx_counts:
        boundary_lookup[(source, pnx)] = count

rows = []
for source, pnx_counts in collapsed_pnx_counts_train.items():
    for pnx, count in pnx_counts:
        with_boundary_count = boundary_lookup.get((source, pnx), 0)
        boundary_pct = (with_boundary_count / count) if count > 0 else 0
        rows.append({
            'pnx': pnx, 
            'source': source, 
            'count': count,
            'with_boundary': with_boundary_count,
            'boundary_pct': round(boundary_pct, 2)
        })

collapsed_pnx_counts_train_df = pd.DataFrame(rows)
collapsed_pnx_counts_train_df.to_csv("collapsed_pnx_counts_train.tsv", index=False, sep="\t")
collapsed_pnx_counts_train_df

,pnx,source,count,with_boundary,boundary_pct
0,.*,Majed,2249,1536,0.68
1,!*,Majed,96,95,0.99
2,**,Majed,32,0,0.00
3,",*",Majed,3,3,1.00
4,"""*",Majed,2,1,0.50
5,@*,Majed,1,1,1.00
6,.*,ArabicMMLU,374,5,0.01
7,**,ArabicMMLU,5,0,0.00
8,)*,ArabicMMLU,3,1,0.33
9,(*,ArabicMMLU,2,0,0.00


In [19]:
consecutive_pnx_counts_dev

defaultdict(<function __main__.<lambda>()>,
            {'Majed': [('<s>. .<s>', 413),
              ('<s>. . .<s>', 32),
              ('<s>! !<s>', 13),
              ('<s>* *<s>', 2),
              ('<s>! ! !<s>', 2),
              ('<s>, , ,<s>', 1),
              ('<s>. . . . .<s>', 1)],
             'ArabicMMLU': [('<s>. . .<s>', 8),
              ('<s>. . . . . . .<s>', 6),
              ('<s>. . . . .<s>', 5),
              ('<s>. . . .<s>', 3),
              ('<s>. .<s>', 2),
              ('<s>. . . . . .<s>', 2),
              ('<s>. . . . . . . . . . . . .<s>', 1),
              ('<s>. . . . . . . . . .<s>', 1),
              ('<s>. . . . . . . . . . . . . . .<s>', 1),
              ('<s>. . . . . . . . . . . . . . . .<s>', 1),
              ('<s>_ _ _ _ _ _<s>', 1)],
             'Emarati Curriculum': [('<s>. .<s>', 6),
              ('<s>. . .<s>', 4),
              ('<s>) )<s>', 2),
              ('<s>( (<s>', 1)],
             'Spacetoon Songs': [('<s>. .<s>', 2)],
    

In [20]:
# Display summary: for each sequence type, show total count vs how many have a boundary (1 label)
print("=== TRAIN SET: Consecutive Punctuations with Sentence Boundaries ===\n")
for source in consecutive_pnx_counts_train:
    print(f"Source: {source}")
    counts_dict = dict(consecutive_pnx_counts_train[source])
    with_bound_dict = dict(consecutive_pnx_with_boundary_train.get(source, []))
    
    for seq, total in consecutive_pnx_counts_train[source]:
        with_bound = with_bound_dict.get(seq, 0)
        without_bound = total - with_bound
        pct = (with_bound / total * 100) if total > 0 else 0
        print(f"  {seq}: {total} total | {with_bound} with boundary ({pct:.1f}%) | {without_bound} without")
    print()

=== TRAIN SET: Consecutive Punctuations with Sentence Boundaries ===

Source: Majed
  <s>. .<s>: 2184 total | 1504 with boundary (68.9%) | 680 without
  <s>! !<s>: 82 total | 81 with boundary (98.8%) | 1 without
  <s>. . .<s>: 60 total | 28 with boundary (46.7%) | 32 without
  <s>* *<s>: 32 total | 0 with boundary (0.0%) | 32 without
  <s>! ! !<s>: 14 total | 14 with boundary (100.0%) | 0 without
  <s>, , ,<s>: 3 total | 3 with boundary (100.0%) | 0 without
  <s>. . . .<s>: 3 total | 2 with boundary (66.7%) | 1 without
  <s>" "<s>: 2 total | 1 with boundary (50.0%) | 1 without
  <s>@ @ @<s>: 1 total | 1 with boundary (100.0%) | 0 without
  <s>. . . . . .<s>: 1 total | 1 with boundary (100.0%) | 0 without
  <s>. . . . . . .<s>: 1 total | 1 with boundary (100.0%) | 0 without

Source: ArabicMMLU
  <s>. . .<s>: 89 total | 0 with boundary (0.0%) | 89 without
  <s>. . . . . . .<s>: 52 total | 0 with boundary (0.0%) | 52 without
  <s>. . . . . . . .<s>: 43 total | 0 with boundary (0.0%) | 43 

In [46]:
# check if there are any keys in consecutive_pnx_counts_dev that are not in consecutive_pnx_counts_train
for source in consecutive_pnx_counts_dev:
    for key in consecutive_pnx_counts_dev[source]:
        if source not in consecutive_pnx_counts_train:
            continue
        flag = False
        for key2 in consecutive_pnx_counts_dev[source]:
            pnx_seq = key2[0]
            if key[0] == pnx_seq:
                flag = True
                break
        if not flag:
            print(f"{source} {key}")
# # check if there are any keys in consecutive_pnx_counts_test that are not in consecutive_pnx_counts_train
# for source in consecutive_pnx_counts_test:
#     for key in consecutive_pnx_counts_test[source]:

In [21]:
collapsed_pnx_counts_train

defaultdict(<function __main__.<lambda>()>,
            {'Majed': [('.*', 2249),
              ('!*', 96),
              ('**', 32),
              (',*', 3),
              ('"*', 2),
              ('@*', 1)],
             'ArabicMMLU': [('.*', 374),
              ('**', 5),
              (')*', 3),
              ('(*', 2),
              ('"*', 2),
              ('_*', 1),
              ('-*', 1),
              ("'*", 1)],
             'Hindawi': [('.*', 103), ('"*', 8), (')*', 2)],
             'Emarati Curriculum': [('.*', 56),
              ('**', 7),
              ('@*', 5),
              ('!*', 2),
              ('(*', 2),
              (')*', 1)],
             'Green Library': [('.*', 713),
              ('<*', 174),
              ('>*', 140),
              ('!*', 23),
              ('"*', 3)],
             'Spacetoon Songs': [('.*', 17)],
             'My Language Sings': [('.*', 28), ('!*', 1), ('"*', 1)],
             'Poems and News': [('.*', 12)],
             'Wikipedia': [(